![Platform+](images/banner.png)

# Lab 9: Unstructured Data + AI
## Governing the Coffee AI System — keep it safe, accurate, and affordable.

> **Persona track: Platform Admin** — same lab, your angle.

:point_right: **In this lesson, you'll ensure the Coffee Kiosk app agent is safe to deploy: customer data is protected, responses are brand-appropriate, and costs are monitored.**

The scenario: You are standing up the Coffee Kiosk AI system for production. As the platform admin you **provision** the AI objects (knowledge base, search service, agent) and then **govern** them — protecting customer data, enforcing brand safety, and monitoring cost — before it goes live on the customer-facing Coffee Kiosk app.

In Lab 5 you governed AI **functions** (redaction, filtering, model cost) on a table. This lab is about governing an AI **agent system** — the retrieval + agent + CoWork stack that powers the customer kiosk. We'll cover:
1. Verify what you provisioned
2. Govern retrieval — grounding and attribute-scoped search
3. Redact customer data from interaction logs
4. Content moderation — brand safety
5. Model selection: cost vs. quality (recap)
6. Monitor AI consumption
7. Build a governed agent wrapper
8. Audit and evaluate agent answers
9. Image analysis governance
10. Ship the agent to production via Snowflake CoWork

A hands-on **CoCo Assist** exercise (build an AI cost monitor) rounds out the lab.

📚 **By the end you'll be able to:**
- Provision and verify an AI agent stack (knowledge base, search service, agent)
- Configure an agent with grounding guardrails, response instructions, and a budget
- Govern retrieval with attribute-scoped search and understand owner's-rights access
- Apply Lab 5's redaction/filtering to an agent's inputs and outputs
- Monitor agent and search consumption
- Grant agent access to a role and verify the boundary
- Audit and evaluate agent answer quality
- Put a governed agent into production via Snowflake CoWork

**Time:** ~45 min · **Lab interface: Workspace Notebook —** Cortex Code (CoCo) is your tutor — ask it to explain or fix anything.
*Source data lives in the shared, read-only `PLATPLUS_DB`; anything you create lives in your own `{{my_db}}`.*

---
👉 **Connecting to your compute service:**
1. In the top bar, if no service is connected, click to connect and select your service.
2. Set your **role and warehouse** using the picker at the top-right — or just run the setup cell below, which sets them for you in SQL. (SQL cells run on the warehouse.)
3. Once the status shows **Connected** (green), run a cell with **Shift+Enter** (run + advance) or **Cmd/Ctrl+Enter** (run this cell only); the **▶** button works too.

> 📌 The service is a container that runs the notebook kernel and your **Python** cells. **SQL cells (and Snowpark pushdown) run on the query warehouse**, not the container. It stays running until it hits its idle timeout or you suspend it manually.
> 💡 **Best practice: Start a fresh chat for this lab.** Before you begin, open a fresh CoCo chat — click **New chat** (the **+** at the top of the panel). Each lab creates its own objects, so a fresh chat ensures CoCo isn't referencing tables or schemas from a prior lab that no longer exist. Shorter conversations also produce sharper answers and use fewer tokens.

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()
user = session.get_current_user().strip('"')
my_db = user + '_DB'
my_wh = user + '_WH'
my_role = user + '_RL'
my_analyst_role = user + '_ANALYST_RL'
print('Context ready. Signed in as ' + user)
print('Your workspace -> db: ' + my_db + ' | wh: ' + my_wh + ' | role: ' + my_role)


## Set your session context

In [ ]:
USE ROLE {{my_role}};
CREATE WAREHOUSE IF NOT EXISTS {{my_wh}}
  WAREHOUSE_SIZE = XSMALL AUTO_SUSPEND = 60 AUTO_RESUME = TRUE INITIALLY_SUSPENDED = TRUE;
USE WAREHOUSE {{my_wh}};
CREATE DATABASE IF NOT EXISTS {{my_db}};
CREATE SCHEMA IF NOT EXISTS {{my_db}}.RAW;


### Provision the AI infrastructure

As the platform admin, **you are responsible for deploying and configuring the AI objects** that other teams and applications depend on. This includes the knowledge base, search service, and agent. These are not engineering experiments — they are governed production services that you provision, monitor, and control access to.

> :point_right: **There is no expectation that you ran the Data Engineer or Data Analyst labs.** Each persona track is independent. The cells below create the AI infrastructure from scratch if it doesn't already exist. If these objects are already in place (e.g., from a prior lab), the SQL safely skips what's already done.

**Why this is an admin task:**
- **You control what gets deployed** — Engineers may prototype in dev, but production AI objects are provisioned by the admin with proper configuration (warehouse sizing, refresh schedules, access grants).
- **Standardization** — You ensure all AI services follow the same naming conventions, are in the correct schema, and use approved models.
- **Governance from day one** — By provisioning these objects yourself, you can verify they meet security requirements before anyone else accesses them.

**What the cells below create:**
| Object | What it does | Admin consideration |
|--------|-------------|--------------------|
| `COFFEE_KNOWLEDGE_BASE` table | Stores parsed PDF content, classifications, and embeddings | You control the schema, retention, and who can write to it |
| `COFFEE_SEARCH_SVC` search service | Indexes content for semantic search | You set the `TARGET_LAG` (freshness vs. cost tradeoff) and the warehouse |
| `{{user}}_COFFEE_MENU_AGENT` agent | Conversational Q&A over the knowledge base | You control who can call it and which model it uses |

**Best practices:**
- Use `CREATE OR REPLACE` for idempotent provisioning — safe to re-run, but note it **rebuilds the object from scratch** (it drops and recreates, discarding any existing rows/data). That's fine for a controlled bootstrap like this lab; in production, prefer incremental refresh or `CREATE ... IF NOT EXISTS` when you must preserve existing data.
- Match `TARGET_LAG` to how often content actually changes — for a weekly-updated menu, a lag of several hours (even a day) is plenty; a shorter lag just refreshes more often than the data changes, adding compute cost with no freshness benefit. Tighten the lag only when the data genuinely updates that fast.
- Use a dedicated warehouse for AI services so you can monitor and limit their compute separately from other workloads

#### Provision the Knowledge Base

The cell below creates the table that stores all parsed coffee drink content and classifications. It then populates it by parsing PDFs from the shared stage with [`AI_PARSE_DOCUMENT`](https://docs.snowflake.com/en/sql-reference/functions/ai_parse_document), then labels each row with [`AI_CLASSIFY`](https://docs.snowflake.com/en/sql-reference/functions/ai_classify) (the same classifier you first met in Lab 4).

**What the SQL does:**
| Step | What happens | Admin consideration |
|------|-------------|--------------------|
| `CREATE OR REPLACE TABLE` | Creates the table fresh (replaces any earlier version with a different schema) | Idempotent — safe to re-run regardless of which persona lab ran previously |
| `INSERT ... FROM DIRECTORY(...)` | Parses all PDFs from the shared stage | Full reload — guarantees consistent state |
| `UPDATE ... WHERE category IS NULL` | Classifies only unclassified rows | Won't overwrite existing classifications |

**Key admin decisions:**
- **Schema placement** — Knowledge base lives in `RAW` schema, separate from application objects
- **Column design** — Includes `parsed_at` timestamp for audit trail and freshness monitoring

**First time seeing these? Quick decoder:**
| Term | What it is |
|------|-----------|
| `DIRECTORY('@...STAGE')` | A queryable listing of the files on a stage — lets you drive set-based processing from SQL (one row per file) |
| `TO_FILE('@stage', path)` | Hands a specific staged file to an AI function as a file reference |
| `AI_PARSE_DOCUMENT(..., {'mode':'LAYOUT'})` | Extracts text (preserving layout) from a PDF so it can be stored and searched |

In [ ]:
-- Bootstrap: Create the knowledge base (OR REPLACE ensures correct schema even if another lab created a narrower version)
CREATE OR REPLACE TABLE {{my_db}}.RAW.COFFEE_KNOWLEDGE_BASE (
    drink_id STRING,
    source_file STRING,
    content STRING,
    category STRING,
    parsed_at TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP()
);

-- Parse all PDFs from the shared stage
INSERT INTO {{my_db}}.RAW.COFFEE_KNOWLEDGE_BASE (drink_id, source_file, content)
SELECT
  REPLACE(SPLIT_PART(d.RELATIVE_PATH, '/', -1), '.pdf', '') AS drink_id,
  d.RELATIVE_PATH AS source_file,
  AI_PARSE_DOCUMENT(
    TO_FILE('@PLATPLUS_DB.ROASTERY_RAW.DOCS_STAGE', d.RELATIVE_PATH),
    {'mode': 'LAYOUT'}
  ):content::STRING AS content
FROM DIRECTORY('@PLATPLUS_DB.ROASTERY_RAW.DOCS_STAGE') d
WHERE d.RELATIVE_PATH LIKE 'coffee_kb/%.pdf';

-- Classify each row
UPDATE {{my_db}}.RAW.COFFEE_KNOWLEDGE_BASE
SET category = AI_CLASSIFY(content, ['Hot Drink', 'Cold Drink', 'Both Hot and Cold']):labels[0]::STRING
WHERE category IS NULL AND content IS NOT NULL;


#### Provision the Search Service

A Cortex Search Service indexes text data and provides fast **hybrid** retrieval (vector + keyword). This is the retrieval layer the agent uses to find relevant content. **Cortex Search generates and manages the vector embeddings for the indexed column itself** — you don't compute or store embeddings by hand. That's why the knowledge base table above has no embedding column.

**Statement breakdown:**
| Clause | What it does | Admin consideration |
|--------|-------------|--------------------|
| `ON content` | The text column to index | Ensure this column contains clean, parsed text |
| `ATTRIBUTES drink_id, category` | Metadata columns for filtered search | Consumers can filter results by these fields |
| `WAREHOUSE = {{my_wh}}` | Compute for index building and refresh | Use a dedicated warehouse so you can monitor AI compute separately |
| `TARGET_LAG = '1 hour'` | Maximum delay before new rows become searchable | Tradeoff: shorter lag = more compute = higher cost |
| `AS (SELECT ...)` | Source query defining what feeds the index | Only includes rows you want searchable |

**Key admin decisions:**
- **TARGET_LAG** — 1 hour is appropriate for content that changes infrequently (menu PDFs). For real-time inventory, you'd use minutes — but at higher cost.
- **Warehouse sizing** — Index refresh uses this warehouse. Monitor its credit consumption and resize if needed.
- **CREATE OR REPLACE** — Rebuilds the index from scratch. In production, prefer incremental (the service auto-refreshes within TARGET_LAG).


In [ ]:
-- Create or replace the search service (idempotent — see the caveat above about rebuilding from scratch)
CREATE OR REPLACE CORTEX SEARCH SERVICE {{my_db}}.RAW.COFFEE_SEARCH_SVC
  ON content
  ATTRIBUTES drink_id, category
  WAREHOUSE = {{my_wh}}
  TARGET_LAG = '1 hour'
  AS (
    SELECT content, drink_id, category, source_file
    FROM {{my_db}}.RAW.COFFEE_KNOWLEDGE_BASE
  );


#### Provision the Agent — with guardrails

A [Cortex Agent](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cortex-agents-manage) combines a search service with an LLM to provide conversational Q&A. This is the object that CoWork (and applications) call to get answers. As the admin, you don't just create it — you **configure its guardrails** in the [specification](https://docs.snowflake.com/en/sql-reference/sql/create-agent):

- **Response instructions** — a plain-English policy the agent must follow on every answer (stay grounded in the knowledge base, stay on-brand, never invent facts, never mention competitors). This is your first line of defence against hallucination and off-brand output.
- **Orchestration instructions** — tell the agent to always ground answers in the CoffeeSearch tool.
- **Budget** — cap the time and tokens the agent may spend per request, so a runaway query can't rack up cost.
- **Column descriptions** — tell the agent what each search column means and which are filterable, which improves retrieval quality.

**Statement breakdown:**
| Part | What it does | Admin consideration |
|------|-------------|--------------------|
| `CREATE OR REPLACE AGENT` | Creates the agent as a first-class Snowflake object | Ownable, grantable, auditable — just like a table or view |
| `COMMENT` | Describes the agent's purpose | Critical for discoverability — other admins and auditors need to know what this does |
| `FROM SPECIFICATION $$ ... $$` | YAML block defining tools and configuration | Version-controllable — store in git alongside your infra-as-code |
| `tools.tool_spec.type: cortex_search` | Tells the agent it can use a search service | You control which data sources the agent can access |
| `tool_resources.max_results: 5` | Limits how many documents the agent retrieves per query | Tradeoff: more results = better answers but higher token cost |

**Key admin decisions:**
- **Access control** — After creation, grant `USAGE ON AGENT` only to roles that need it
- **Model governance** — The agent uses Snowflake's default LLM. If you need a specific model, configure it in the specification
- **Audit trail** — Agent calls are logged in ACCOUNT_USAGE, so you can track who's using it and how much it costs

**Best practice:** Treat agents like any other production service — document them, control access via roles, and monitor their consumption in Section 5.

In [ ]:
-- Bootstrap: Create the governed agent (with guardrails)
CREATE OR REPLACE AGENT {{my_db}}.RAW.{{user}}_COFFEE_MENU_AGENT
  COMMENT = 'Coffee menu Q&A agent for the customer kiosk'
  PROFILE = '{"display_name": "Coffee Menu Expert ({{user}})"}'  -- includes your username so you can find it in CoWork
  FROM SPECIFICATION
  $$
  orchestration:
    budget:
      seconds: 30
      tokens: 16000

  instructions:
    response: "You are a friendly coffee shop assistant. Answer only using the coffee knowledge base. Keep answers concise and professional. If the knowledge base does not contain the answer, say you don't have that information rather than guessing. Never mention competitors."
    orchestration: "Always use the CoffeeSearch tool to ground answers in the coffee knowledge base."
    sample_questions:
      - question: "Which drinks have the most caffeine?"
      - question: "What is the difference between a flat white and a latte?"

  tools:
    - tool_spec:
        type: "cortex_search"
        name: "CoffeeSearch"
        description: "Searches the coffee knowledge base for drink information, preparation methods, and menu details."

  tool_resources:
    CoffeeSearch:
      name: "{{my_db}}.RAW.COFFEE_SEARCH_SVC"
      max_results: "5"
      id_column: "drink_id"
      columns_and_descriptions:
        content:
          description: "Full parsed text describing a coffee drink: ingredients, preparation, caffeine, flavor."
          type: "string"
          searchable: true
          filterable: false
        category:
          description: "Drink temperature category. Values: Hot Drink, Cold Drink, Both Hot and Cold."
          type: "string"
          searchable: false
          filterable: true
        drink_id:
          description: "Short drink identifier, e.g. latte, cold_brew, espresso."
          type: "string"
          searchable: false
          filterable: true
  $$;


## 1 · Verify what you provisioned
Before layering governance on top, confirm the stack you just provisioned is complete and healthy: the data source, the search index, and the agent.

**Why this matters (platform admin perspective):**
- **You own the go/no-go decision** — You provisioned it; now you confirm it meets production standards before customers see it.
- **Accountability requires visibility** — If something goes wrong (hallucinations, missing content, stale data), you need to know what was deployed and who owns each component.
- **Compliance documentation** — Many organizations require a pre-deployment audit trail showing what AI systems are in production, what data they access, and what models they use.

In [ ]:
-- Check the knowledge base table
SELECT drink_id, category, LENGTH(content) AS content_chars, parsed_at
FROM {{my_db}}.RAW.COFFEE_KNOWLEDGE_BASE
ORDER BY drink_id;


In [ ]:
-- Verify the search service exists and is healthy
SHOW CORTEX SEARCH SERVICES IN SCHEMA {{my_db}}.RAW;


In [ ]:
-- Verify the agent exists
SHOW AGENTS IN SCHEMA {{my_db}}.RAW;


> :mag_right: **Audit checklist:**
> - Knowledge base has one row per coffee PDF — 8 drinks (or 9 if you also completed the Data Engineer lab, which adds matcha_latte)
> - All documents have non-null content (parsing succeeded)
> - Search service is active with a reasonable TARGET_LAG
> - Agent is connected to the correct search service
>
> If any of these fail, fix the provisioning cells above before moving on to governance.

## 2 · Govern retrieval — grounding & attribute scope
The agent answers by **retrieving** from the search service, then having the LLM answer *only* from what it retrieved — that's RAG ([retrieval-augmented generation](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cortex-search/cortex-search-overview)). Two governance levers matter here:

- **Grounding** — answers should come from your knowledge base, not the model's imagination. You'll verify this directly below.
- **Attribute scope** — the search service exposes `ATTRIBUTES` (here `drink_id`, `category`) that let you *filter* what can be retrieved.

The cell below previews retrieval directly (no LLM) and **filters to one category** using the [`@eq` filter](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cortex-search/query-cortex-search-service). This is how you scope what an agent is allowed to surface.

> 📌 **`SEARCH_PREVIEW` is a testing tool.** We use it here to inspect retrieval directly — no LLM, no agent, just the raw search results. In production the agent calls the search service automatically (it's configured as a `tool_resource`). For programmatic access outside an agent, use the [Python SDK](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cortex-search/query-cortex-search-service#python-sdk) or REST API — not `SEARCH_PREVIEW`.

In [ ]:
-- Retrieve straight from the search index, scoped to one category
SELECT f.value:drink_id::STRING AS drink_id, f.value:category::STRING AS category
FROM TABLE(FLATTEN(input => PARSE_JSON(
  SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
    '{{my_db}}.RAW.COFFEE_SEARCH_SVC',
    '{
       "query": "which drink is smooth and low in caffeine",
       "columns": ["drink_id", "category"],
       "filter": {"@eq": {"category": "Cold Drink"}},
       "limit": 5
     }'
  )):results
)) f;

> :mag_right: **What to notice:** every returned row is a `Cold Drink` — the `@eq` filter scoped retrieval to that attribute. Swap `"Cold Drink"` for `"Hot Drink"` to see the scope change. Other operators: `@and`, `@or`, `@not`, `@gte`, `@lte`, `@contains`.
>
> :warning: **Owner's-rights caution (important governance point):** a Cortex Search service runs with **owner's rights**. Any role you grant `USAGE` on the service can retrieve **everything the service indexed** — regardless of that role's grants on the underlying table. Grant search/agent access deliberately: it is a grant over *all* the indexed data, not a row-level view.

## 3 · Redact customer data from interaction logs
> :point_right: **Recap from Lab 5.** You already used [`AI_REDACT`](https://docs.snowflake.com/en/sql-reference/functions/ai_redact) to strip PII and built it into a governed *view*. Here the surface is different: **live agent interaction logs**. Same function, new target — so we'll keep this brief and move on to the agent-specific governance.

When customers interact with the kiosk, their questions may contain PII (names, emails, phone numbers). Before storing interaction logs for analytics, strip PII with AI_REDACT.

**Why this matters (platform admin perspective):**
- **Regulatory compliance** — GDPR, CCPA, and other regulations require that PII is not stored without consent. Interaction logs used for analytics must be anonymized.
- **Data breach risk reduction** — If your analytics tables are ever exposed, redacted logs contain no sensitive customer data.
- **Enabling safe analytics** — Product teams want to analyze what customers ask, but they don't need to know *who* asked. Redaction gives them the insights without the liability.

Let's simulate some customer interactions and apply `AI_REDACT`.

In [ ]:
-- Simulated customer interaction logs
CREATE OR REPLACE TEMPORARY TABLE {{my_db}}.RAW.AGENT_INTERACTION_LOG AS
SELECT * FROM VALUES
  (1, 'Hi, my name is Sarah Chen and I want something with less caffeine. My email is sarah.chen@gmail.com', CURRENT_TIMESTAMP()),
  (2, 'What is a flat white? I am Jake from the downtown office, phone 555-0142', CURRENT_TIMESTAMP()),
  (3, 'Can you recommend something sweet? I have a nut allergy. Contact me at maria.garcia@company.com', CURRENT_TIMESTAMP()),
  (4, 'Just give me your strongest espresso drink please', CURRENT_TIMESTAMP()),
  (5, 'Hi this is Dr. Robert Williams, ID 4471. What pairs with chocolate?', CURRENT_TIMESTAMP())
AS t(interaction_id, customer_query, timestamp);


In [ ]:
-- Apply AI_REDACT to strip PII
SELECT
  interaction_id,
  customer_query AS original_query,
  AI_REDACT(customer_query) AS redacted_query
FROM {{my_db}}.RAW.AGENT_INTERACTION_LOG;


> :warning: **Your results may look different to what's described in this lab — that's normal.**
>
> LLMs are **non-deterministic** — even with the same model, prompt, and data, each run can produce slightly different output. This is by design (the model's `temperature` parameter introduces controlled randomness).
>
> **What this means for you:** Focus on the *pattern* and structure of results rather than matching exact text. If your output is structured correctly but worded differently, that's expected.

> :mag_right: **Review.** PII like names, emails, and phone numbers should be replaced with placeholders ([PERSON], [EMAIL], [PHONE]). Row 4 has no PII and should pass through unchanged.

## 4 · Content moderation — brand safety
> :point_right: **Recap from Lab 5.** You used [`AI_FILTER`](https://docs.snowflake.com/en/sql-reference/functions/ai_filter) as a plain-English boolean test. Here you apply it to a new surface: the **agent's own responses**, before they reach a customer.

Before the agent's responses reach customers, verify they're appropriate. Use AI_FILTER as a post-processing gate on agent output.

**Why this matters (platform admin perspective):**
- **Brand reputation** — One inappropriate response on the Coffee Kiosk app screenshot going viral can damage the brand. Content moderation is your safety net.
- **Liability reduction** — If the agent generates harmful, misleading, or offensive content, the organization is liable. Filtering catches these before they reach customers.
- **Consistent customer experience** — Ensure all responses maintain the same professional, helpful tone regardless of what the LLM generates.

**How `AI_FILTER` works:** You provide the text and a natural-language description of what should be filtered. It returns TRUE (safe) or FALSE (should be blocked). In production, you'd wrap this around every agent response before displaying it.

In [ ]:
-- Test agent responses for brand safety
WITH agent_responses AS (
  SELECT 'Our espresso is made with premium single-origin beans roasted in small batches.' AS response
  UNION ALL
  SELECT 'I dont know, just drink whatever. Coffee is coffee.' AS response
  UNION ALL
  SELECT 'The competitor across the street uses cheaper beans but charges the same price.' AS response
  UNION ALL
  SELECT 'Our cold brew is steeped for 24 hours to achieve a smooth, naturally sweet flavor profile.' AS response
)
SELECT
  LEFT(response, 80) AS response_preview,
  AI_FILTER(PROMPT('This response is professional, helpful, and appropriate for a customer-facing coffee shop assistant: {0}', response)) AS is_brand_safe
FROM agent_responses;


> :mag_right: **How to read the results:**
>
> | IS_BRAND_SAFE | What it means | Action in production |
> |--------------|---------------|---------------------|
> | `TRUE` (1) | The response passes your filter criteria — it's professional, helpful, and appropriate | Show to customer |
> | `FALSE` (0) | The response fails — it's dismissive, mentions competitors, or is otherwise off-brand | Block it and show a fallback message (e.g., "Let me find a better answer for you") |
>
> **Expected results for this test data:**
> - Row 1 (espresso description) → TRUE — professional and on-topic
> - Row 2 ("just drink whatever") → FALSE — dismissive, unprofessional tone
> - Row 3 (competitor mention) → FALSE — references competition, off-brand
> - Row 4 (cold brew description) → TRUE — professional and on-topic
>
> **Governance insight:** In production, you'd add this filter as a post-processing step on the agent's output before it reaches the Coffee Kiosk app screen. If `AI_FILTER` returns FALSE, the app shows a generic fallback instead of the agent's response.

### :jigsaw: Your turn — write the brand-safety rule
The power of `AI_FILTER` is that *you* define the policy in plain English. In the cell below, the condition is blanked out (`______`). Replace it with a natural-language rule that is TRUE only when a response is safe to show a customer (professional, on-topic for coffee, no competitor mentions). The `: {0}` that feeds in the response is already there — you only write the condition text. Run it, then reveal the reference if you get stuck.

In [ ]:
-- Replace ______ with your plain-English brand-safety rule
WITH agent_responses AS (
  SELECT 'Our cold brew steeps for 24 hours for a smooth, naturally sweet flavor.' AS response
  UNION ALL
  SELECT 'Honestly the place across the street is cheaper, go there.' AS response
)
SELECT
  LEFT(response, 70) AS response_preview,
  AI_FILTER(PROMPT('______: {0}', response)) AS is_brand_safe   -- 🧩 write just the condition
FROM agent_responses;

<details>
<summary>🔧 Reveal a working rule</summary>

```sql
AI_FILTER(PROMPT('This is a professional, on-brand coffee shop response that does not mention or recommend competitors: {0}', response)) AS is_brand_safe
```
Your wording can differ — `AI_FILTER` judges meaning, not exact words. The competitor-mentioning row should return FALSE.
</details>

## 5 · Model selection: cost vs. quality (recap)
> :point_right: **Recap from Lab 5.** You already compared `llama3.1-8b` vs `llama3.1-70b` and used `AI_COUNT_TOKENS` to see that model *size* sets the price-per-token, not the token count. That analysis applies directly here.

**The agent-specific decision:** your agent's answers are **customer-facing**, so quality matters — approve a capable model for it. Reserve the cheaper model for internal, constrained tasks (like the `AI_CLASSIFY` labelling in provisioning). You set the agent's model in its specification (the `models:` field); leaving it unset lets Snowflake pick a strong default.

## 6 · Monitor AI consumption
Track who's using AI functions and how much they're costing.

**Why this matters (platform admin perspective):**
- **Cost visibility** — AI functions consume credits. Without monitoring, costs can spike unexpectedly (e.g., a runaway Task calling AI_COMPLETE in a loop).
- **Chargeback** — If multiple teams share an account, you need to attribute AI costs to the correct cost center.
- **Anomaly detection** — A sudden spike in AI credit usage could indicate a bug, an inefficient query, or unauthorized usage.

In [ ]:
-- Check recent Cortex AI usage (last 7 days)
SELECT
  USAGE_DATE,
  SERVICE_TYPE,
  SUM(CREDITS_USED) AS total_credits
FROM SNOWFLAKE.ACCOUNT_USAGE.METERING_DAILY_HISTORY
WHERE (SERVICE_TYPE ILIKE 'AI_%' OR SERVICE_TYPE ILIKE '%CORTEX%')
  AND USAGE_DATE >= DATEADD('day', -7, CURRENT_DATE())
GROUP BY USAGE_DATE, SERVICE_TYPE
ORDER BY USAGE_DATE DESC;


> :mag_right: **How to read these results:**
>
> | SERVICE_TYPE | What it represents | What generates it |
> |-------------|-------------------|------------------|
> | `AI_FUNCTIONS` | Direct calls to AI functions (AI_COMPLETE, AI_CLASSIFY, AI_EMBED, AI_FILTER, etc.) | Your SQL queries in this lab, the agent's internal LLM calls |
> | `CORTEX_AGENTS` | Credits consumed by Cortex Agent calls | Every time someone queries your agent (via SQL or CoWork) |
> | `CORTEX_SEARCH` | Credits for search service index maintenance and queries | Search service refresh cycles + query processing |
> | `CORTEX_CODE_SNOWSIGHT` | CoCo (Cortex Code) usage in the browser | Your CoCo interactions in Snowsight |
> | `SNOWFLAKE_COCO` | CoCo (Cortex Code) Desktop app usage | The standalone CoCo Desktop application |
> | `AI_SERVICES` | Other AI services (document parsing, etc.) | [`AI_PARSE_DOCUMENT`](https://docs.snowflake.com/en/sql-reference/functions/ai_parse_document) and similar |
>
> **What to look for as an admin:**
> - **Unexpectedly high values** — Could indicate a runaway query, a Task loop, or unauthorized usage
> - **Service types you didn't expect** — If `CORTEX_AGENTS` shows usage but you haven't deployed an agent, someone else may have
> - **Day-over-day trends** — Stable is good. Sudden spikes warrant investigation
> - **Ratio of agent vs function costs** — Agents bundle search + LLM, so they're more expensive per call than simple AI functions
>
> *Note: If some rows show 0 or no data appears, this is normal — ACCOUNT_USAGE has up to 2 hours of latency. Also, `SNOWFLAKE_COCO` won't appear in this query because the filter matches `AI_%` and `%CORTEX%` — to include CoCo Desktop, add `OR SERVICE_TYPE = 'SNOWFLAKE_COCO'` to the WHERE clause.*

In [ ]:
-- Who is running AI queries?
SELECT
  user_name,
  COUNT(*) AS ai_query_count
FROM SNOWFLAKE.ACCOUNT_USAGE.QUERY_HISTORY
WHERE (query_text ILIKE '%AI_%' OR query_text ILIKE '%CORTEX%')
  AND start_time >= DATEADD('day', -7, CURRENT_TIMESTAMP())
GROUP BY user_name
ORDER BY ai_query_count DESC
LIMIT 10;


> :mag_right: **How to read these results:**
>
> This shows which users ran queries containing AI function calls in the last 7 days.
>
> | Column | What it means |
> |--------|---------------|
> | `USER_NAME` | The Snowflake user who executed the query |
> | `AI_QUERY_COUNT` | Number of queries that referenced AI/Cortex functions |
>
> **What to look for as an admin:**
> - **Expected users** — Are the people using AI functions the ones you authorized? If an unknown user appears, investigate.
> - **Disproportionate usage** — One user with 10x more queries than others may indicate automated/looping calls (e.g., a runaway script) or simply heavy development activity.
> - **Service accounts** — If a Task or pipeline runs AI functions, it appears under its owner's user. Consider using dedicated service users for automated workloads so you can separate human vs. machine usage.
>
> **Limitations:**
> - This uses `ILIKE '%AI_%'` on query text, which is a rough filter — it may catch queries that *mention* AI functions in comments without actually calling them.
> - ACCOUNT_USAGE views have **up to 2 hours of latency**. Very recent queries may not appear yet.
> - For precise AI credit attribution, use `METERING_DAILY_HISTORY` (Section 5 above) rather than query counting.
>
> **Best practice:** In production, set up resource monitors or alerts on credit thresholds. Don't rely on manual checks — automate notifications at 50%, 75%, and 90% of your AI budget.

## 7 · Build a governed agent wrapper
Create a workflow that wraps the agent interaction with guardrails: redact input PII, moderate output, log everything.

> 📌 **Why a wrapper and not a native agent feature?** As of today, Snowflake agents don't have built-in pre/post-processing pipelines for redaction or moderation. The stored-procedure wrapper pattern here is how admins add those layers — think of it as middleware, the same approach you'd use to add logging or rate-limiting to any API. You build it once, every app routes through it, and when native guardrails land in the product, you swap out the wrapper without changing the apps.

**Why this matters (platform admin perspective):**
- **Defense in depth** — No single control is sufficient. The governed wrapper chains multiple protections: PII redaction → agent call → content moderation → logging. If one layer fails, others still protect.
- **Audit trail** — Every interaction is logged (with PII redacted) so you can investigate issues, measure quality, and demonstrate compliance.
- **Single enforcement point** — Instead of relying on every app developer to implement guardrails, you build them once in the wrapper. All apps go through the same governance layer.

In [ ]:
-- Create a governed interaction table
CREATE OR REPLACE TABLE {{my_db}}.RAW.GOVERNED_INTERACTIONS (
  interaction_id NUMBER AUTOINCREMENT,
  raw_query STRING,
  redacted_query STRING,
  agent_response STRING,
  is_brand_safe BOOLEAN,
  model_used STRING DEFAULT 'llama3.1-70b',
  created_at TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP()
);


#### What the next cell does (step by step):

The "Simulate Governed Interaction" cell below chains all the governance layers into a single SQL statement. Here's the flow:

```
Customer input → AI_REDACT → AI_COMPLETE → AI_FILTER → INSERT into log table
```

| Step | CTE name | What happens | Why |
|------|----------|-------------|-----|
| 1 | `input` | Simulates a customer question containing PII (name + email) | Real customers type personal info into chat |
| 2 | `redacted` | `AI_REDACT` strips PII before it goes anywhere | Compliance — never pass raw PII to the LLM |
| 3 | `answered` | [`AI_COMPLETE`](https://docs.snowflake.com/en/sql-reference/functions/ai_complete) generates an answer using the redacted query + knowledge base | The agent responds to the cleaned question |
| 4 | Final SELECT | `AI_FILTER` checks if the response is brand-safe | Safety gate before the response reaches the customer |
| 5 | INSERT | Everything gets logged to `GOVERNED_INTERACTIONS` table | Audit trail — every interaction recorded (PII-free) |

**This is the governed wrapper pattern in action.** In production, this entire flow would be a stored procedure that every application calls instead of the raw agent.

In [ ]:
-- Simulate a governed interaction flow
INSERT INTO {{my_db}}.RAW.GOVERNED_INTERACTIONS (raw_query, redacted_query, agent_response, is_brand_safe)
WITH input AS (
  SELECT 'Hi, I am Sarah Chen (sarah@email.com). What is your smoothest coffee?' AS query
),
redacted AS (
  SELECT query, AI_REDACT(query) AS clean_query FROM input
),
answered AS (
  SELECT
    query,
    clean_query,
    AI_COMPLETE('llama3.1-70b',
      'You are a friendly coffee expert. Answer concisely: ' || clean_query
      || '\nContext: ' || (SELECT LISTAGG(content, '\n') FROM {{my_db}}.RAW.COFFEE_KNOWLEDGE_BASE)
    ) AS response
  FROM redacted
)
SELECT
  query,
  clean_query,
  response,
  AI_FILTER(PROMPT('This response is professional and appropriate for a coffee shop: {0}', response)) AS safe
FROM answered;


In [ ]:
-- Review the governed interaction
SELECT * FROM {{my_db}}.RAW.GOVERNED_INTERACTIONS;


> :mag_right: **How to read the governed interaction results:**
>
> The "Review Governed Interactions" cell shows what was logged. Each row represents one customer interaction with these columns:
>
> | Column | What to check | What "correct" looks like |
> |--------|--------------|---------------------------|
> | `RAW_QUERY` | The original customer input | Contains PII (name, email) — this is what the customer typed |
> | `REDACTED_QUERY` | After AI_REDACT processed it | PII replaced with placeholders like `[PERSON]`, `[EMAIL]` — the question is intact but anonymized |
> | `AGENT_RESPONSE` | The AI's answer | A relevant coffee recommendation based on the redacted question |
> | `IS_BRAND_SAFE` | AI_FILTER's verdict | `TRUE` = safe to show customer. `FALSE` = would need to be blocked |
>
> **The governance flow in summary:**
> 1. Customer asks a question (may contain PII)
> 2. `AI_REDACT` strips PII before processing
> 3. `AI_COMPLETE` generates a response using the redacted input
> 4. `AI_FILTER` checks the response is brand-safe
> 5. If safe: show to customer. If not: show a fallback message
> 6. Log everything (redacted) for analytics and audit
>
> **Key insight:** The `RAW_QUERY` column still contains PII in this demo because we stored it for comparison purposes. In a real production system, you would **never store the raw query** — only the redacted version. The raw input would be discarded immediately after redaction.

## 8 · Audit & evaluate the agent
Governing an agent doesn't stop at guardrails — you must be able to **prove** it behaves. Two admin practices:

- **Audit log** — record every agent answer (question, response, timestamp) so you can investigate complaints and demonstrate compliance.
- **Automated evaluation** — score answers for grounding/quality with an LLM-as-judge, so you catch drift without reading every response by hand.

In [ ]:
-- Audit log for agent answers
CREATE OR REPLACE TABLE {{my_db}}.RAW.AGENT_ANSWER_LOG (
  asked_at TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP(),
  question STRING,
  answer STRING,
  grounding_score STRING,
  looks_trustworthy BOOLEAN
);

#### Evaluate answers with an LLM-as-judge
The cell below takes two candidate answers — one grounded, one absurd — and scores each two ways: an [`AI_COMPLETE`](https://docs.snowflake.com/en/sql-reference/functions/ai_complete) 1–5 grounding score, and an `AI_FILTER` trustworthiness gate. In production you'd run this over a sample of real agent answers and alert on low scores. (Keeping the judge condition simple matters — an over-specified condition produces false negatives.)

In [ ]:
-- LLM-as-judge evaluation of agent answers
INSERT INTO {{my_db}}.RAW.AGENT_ANSWER_LOG (question, answer, grounding_score, looks_trustworthy)
WITH candidate_answers AS (
  SELECT 'Which drink has the most caffeine?' AS question,
         'Cold brew has the most caffeine at 200-250mg per 16oz because it steeps 12-24 hours.' AS answer
  UNION ALL
  SELECT 'Recommend a drink' AS question,
         'Our pumpkin unicorn frappe has 900mg of caffeine and cures headaches.' AS answer
)
SELECT
  question,
  answer,
  AI_COMPLETE('llama3.1-70b', 'Rate this coffee assistant answer for factual grounding from 1 to 5 (5=fully grounded, 1=hallucinated). Reply ONLY the number: ' || answer) AS grounding_score,
  AI_FILTER(PROMPT('This is a realistic, professional coffee shop answer with no absurd or fabricated claims: {0}', answer)) AS looks_trustworthy
FROM candidate_answers;

In [ ]:
-- Review the evaluation log
SELECT question, LEFT(answer, 70) AS answer_preview, grounding_score, looks_trustworthy
FROM {{my_db}}.RAW.AGENT_ANSWER_LOG
ORDER BY asked_at DESC;

> :mag_right: **How to read it:** the grounded answer scores ~4–5 and `looks_trustworthy = TRUE`; the absurd one scores ~1 and `FALSE`. This is a lightweight guardrail you can automate — log every agent answer, judge it, and alert when scores drop. For richer evaluation (relevance, correctness against a reference set) Snowflake offers Cortex Agent evaluations / AI Observability.

## 9 · Image analysis governance
The Coffee Kiosk app may also let customers photograph a drink for identification. This requires multimodal AI (models that understand images). As admin, you decide which models are approved for this capability.

**Why this matters (platform admin perspective):**
- **Model capability boundaries** — Not all models can process images. Approving the wrong model for an image task means silent failures or errors in production.
- **Cost control** — Multimodal models (that process images/video) are typically more expensive per call than text-only models. You need to approve their use explicitly.
- **Data sensitivity** — Images may contain customer faces, location data, or other sensitive information. You need to decide what's acceptable to send to which model.

Before running the analysis, look at the exact image the model will see — that way you can judge whether its description is accurate.

In [ ]:
# Display the image the model will analyze, so you can judge the AI's accuracy
from IPython.display import display, HTML
import base64

session.file.get('@PLATPLUS_DB.ROASTERY_RAW.DOCS_STAGE/coffee_images/cappuccino.jpg', '/tmp/')
with open('/tmp/cappuccino.jpg', 'rb') as f:
    img_b64 = base64.b64encode(f.read()).decode()
display(HTML(
    f'<h4>cappuccino.jpg — what the model will see</h4>'
    f'<img src="data:image/jpeg;base64,{img_b64}" width="360">'
))

In [ ]:
-- Test image analysis with an approved model
SELECT AI_COMPLETE(
  'pixtral-large',
  'Identify this coffee drink. What is it and what are its key characteristics?',
  TO_FILE('@PLATPLUS_DB.ROASTERY_RAW.DOCS_STAGE', 'coffee_images/cappuccino.jpg')
) AS image_analysis;


> :bulb: **Model approval policy for image analysis:**
> - `pixtral-large` — Approved for drink identification (multimodal)
> - `gemini-3.1-pro` — Approved for video analysis (multimodal)
> - Text-only models (llama, etc.) — NOT approved for image/video tasks
>
> Document this in your team’s AI governance runbook.

---
## :robot_face: CoCo Assist — Build an AI Cost Monitor

Your security team asks: *"Can you set up something that watches daily Cortex AI credits and logs a warning when usage exceeds a threshold?"*

Instead of writing this yourself, let's use **CoCo** to generate the SQL.

**Why a Task-based monitor (not a Snowflake Alert):**

Snowflake Alerts and Resource Monitors require `ACCOUNTADMIN` privileges. In most organizations, platform admins don't have — and shouldn't need — account-level access for routine monitoring. Instead, the common production pattern is:
1. A **table** to log threshold violations
2. A **Task** that periodically checks `METERING_DAILY_HISTORY` and inserts a warning row when usage exceeds your threshold
3. Downstream processes (dashboards, email integrations) that read the alert table

This approach works with standard role privileges, is auditable, and can be customized per team or cost center.

**What this prompt asks CoCo to build (in plain language):**
- A table (`AI_COST_ALERTS`) to store any threshold violations
- A Task that runs every 2 minutes (for lab testing — production would be hourly or daily), checks yesterday's and today's Cortex AI credits from `METERING_DAILY_HISTORY`, and inserts a warning row if the total exceeds 0.01 credits

> :warning: **About the lab-friendly threshold (0.01 credits):** We use a very low threshold so even a fresh account that just ran a few AI functions in this lab will trigger an alert — proving the monitor works. In production, you'd set this to your actual budget threshold (e.g., 50 or 100 credits per day). The pattern is the same regardless of the number.

> :warning: **About the 2-minute schedule:** We use a short interval so you can quickly verify the Task runs. In production, cost monitoring typically runs hourly or daily — not every 2 minutes. After testing, we'll suspend the Task.

**Jargon decoder:**
| Term in the prompt | What it means |
|---|---|
| `Snowflake Task` | A scheduled job that runs SQL on a timer |
| `METERING_DAILY_HISTORY` | A system view that tracks daily credit consumption by service type |
| `0.01 credits per day` | A deliberately low threshold for lab testing (production would be higher) |
| `AI_COST_ALERTS table` | Where violations get logged — queryable, auditable, and can trigger downstream actions |
| `SERVICE_TYPE ILIKE '%AI%' OR '%CORTEX%'` | Filters for AI-related credit consumption only |

🤖 **CoCo Prompt** — *click to copy, then paste into the CoCo panel:*

<pre style="user-select: all; -webkit-user-select: all; background: rgba(41, 181, 232, 0.12); border-left: 4px solid #29B5E8; padding: 12px; border-radius: 0 4px 4px 0; cursor: pointer; white-space: pre-wrap; word-wrap: break-word;">Write SQL that: 1) Creates a table called {{my_db}}.RAW.AI_COST_ALERTS with columns alert_time TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP(), usage_date DATE, total_credits NUMBER(38,9), threshold NUMBER(38,9), and message STRING. 2) Creates a Task called {{my_db}}.RAW.AI_COST_MONITOR_TASK using warehouse {{my_wh}} that runs every 2 minutes. The Task should check SNOWFLAKE.ACCOUNT_USAGE.METERING_DAILY_HISTORY for rows where SERVICE_TYPE ILIKE '%AI%' OR SERVICE_TYPE ILIKE '%CORTEX%' and USAGE_DATE >= CURRENT_DATE() - 1. If the SUM of CREDITS_USED for any single day exceeds 0.01, INSERT a warning row into AI_COST_ALERTS with the usage_date, total_credits, threshold of 0.01, and a message like 'Daily AI credits exceeded threshold'. Avoid duplicate alerts by checking the date isn't already in the alerts table. Include ALTER TASK RESUME and SUSPEND commands. Do NOT run this code — only generate it so I can copy and paste it into my notebook cell and run it myself.</pre>

**What to do with the result:**
> 1. Copy CoCo's generated SQL
> 2. Paste it into the empty cell below
> 3. Run it
> 4. If it errors, paste the error message back to CoCo — it will fix it
> 5. Wait ~2 minutes, then run the verification cells to check if the Task logged any alerts
> 6. **Suspend the Task** when done

**A note on prompting:** Prompts are iterative. If CoCo generates something that doesn't quite work, refine your prompt or paste the error back. This back-and-forth is how real engineering with AI assistants works. If you get stuck or want to see what a working solution looks like, expand the **"Resolution - Cost Monitor"** cell below (it's collapsed by default).

This iterate-with-CoCo workflow is how admins build monitoring without memorizing Task syntax.

In [ ]:
-- Paste CoCo's suggestion on the blank line below, then run it


<details>
<summary>🔧 <b>Resolution — AI Cost Monitor</b> (click to expand a working solution, then copy it into the empty cell above)</summary>

```sql
-- RESOLUTION: AI Cost Monitor. This is what CoCo should generate from the prompt above.

-- Step 1: Create the alerts table
CREATE TABLE IF NOT EXISTS {{my_db}}.RAW.AI_COST_ALERTS (
    alert_time TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP(),
    usage_date DATE,
    total_credits NUMBER(38,9),
    threshold NUMBER(38,9),
    message STRING
);

-- Step 2: Create the monitoring Task (scheduled — a monitor runs on a timer)
CREATE OR REPLACE TASK {{my_db}}.RAW.AI_COST_MONITOR_TASK
  WAREHOUSE = {{my_wh}}
  SCHEDULE = '2 MINUTE'
AS
BEGIN
  INSERT INTO {{my_db}}.RAW.AI_COST_ALERTS (usage_date, total_credits, threshold, message)
  SELECT
    USAGE_DATE,
    SUM(CREDITS_USED) AS total_credits,
    0.01 AS threshold,
    'Daily AI credits exceeded threshold'
  FROM SNOWFLAKE.ACCOUNT_USAGE.METERING_DAILY_HISTORY
  WHERE (SERVICE_TYPE ILIKE '%AI%' OR SERVICE_TYPE ILIKE '%CORTEX%')
    AND USAGE_DATE >= CURRENT_DATE() - 1
  GROUP BY USAGE_DATE
  HAVING SUM(CREDITS_USED) > 0.01
    AND USAGE_DATE NOT IN (SELECT usage_date FROM {{my_db}}.RAW.AI_COST_ALERTS);
END;

-- Start the Task
ALTER TASK {{my_db}}.RAW.AI_COST_MONITOR_TASK RESUME;

-- Run this AFTER testing to stop the Task:
-- ALTER TASK {{my_db}}.RAW.AI_COST_MONITOR_TASK SUSPEND;
```

</details>

### Verify the cost monitor ran

After creating and resuming the Task, wait about 2 minutes for it to fire, then run the cells below.

> 💡 **While you wait:** read ahead to Section 10 (The Production Kiosk) — it's all markdown until the test cell, so you'll have context ready when you get there. Come back here once ~2 minutes have passed.

**How to verify:**
1. **Check Task History** — Look for `state = SUCCEEDED` to confirm the Task executed
   - If you see `SCHEDULED` or `EXECUTING` — wait another minute and re-run
   - Once you see `SUCCEEDED`, check the alerts table
2. **Check AI Cost Alerts** — If any day exceeded the 0.01-credit threshold (likely, given all the AI functions we've run today), you'll see a row with the date, credit total, and threshold
   - If empty, the threshold wasn't breached — that's also a valid outcome (it means daily AI usage stayed under 0.01 credits)

> :warning: **Timing note:** `METERING_DAILY_HISTORY` has up to 2 hours of latency. The Task may run successfully but find no data yet if you only started using AI functions recently. In production, this latency means your alerts fire with a delay — which is acceptable for daily cost monitoring.

**Don't forget to suspend the Task when done:**

In [ ]:
-- Check Task execution history
SELECT name, state, scheduled_time, completed_time
FROM TABLE({{my_db}}.INFORMATION_SCHEMA.TASK_HISTORY(
  TASK_NAME => 'AI_COST_MONITOR_TASK',
  SCHEDULED_TIME_RANGE_START => DATEADD(hour, -1, CURRENT_TIMESTAMP())
))
ORDER BY scheduled_time DESC
LIMIT 5;


In [ ]:
-- Check if any cost alerts were logged
SELECT * FROM {{my_db}}.RAW.AI_COST_ALERTS
ORDER BY alert_time DESC;


> :mag_right: **How to read the alerts:**
>
> | Column | What it means |
> |--------|---------------|
> | `ALERT_TIME` | When the Task detected the threshold breach |
> | `USAGE_DATE` | The day that exceeded the threshold |
> | `TOTAL_CREDITS` | Actual credits consumed that day (across all AI/Cortex services) |
> | `THRESHOLD` | The limit you set (0.01 credits — deliberately low for lab testing) |
> | `MESSAGE` | Human-readable description of the violation |
>
> **Why you might see multiple rows:** The Task checks the last 2 days of usage (`CURRENT_DATE() - 1`). If multiple days exceeded the threshold, you'll get one alert row per day — all logged in a single Task execution. The `NOT IN` check prevents duplicates on subsequent runs.
>
> **If you see rows:** Your daily AI usage exceeded 0.01 credits — which is expected since you've been running AI functions throughout this lab. This proves the monitor is working correctly. In production with a higher threshold (e.g., 50 credits), this table would only show actual budget concerns.
>
> **If the table is empty:** `METERING_DAILY_HISTORY` has up to 2 hours of latency. If you just started using AI functions, data may not appear yet. Wait a few minutes and re-run. In production, this latency is acceptable for daily cost monitoring.

In [ ]:
-- IMPORTANT: Suspend the Task to stop scheduled runs
ALTER TASK {{my_db}}.RAW.AI_COST_MONITOR_TASK SUSPEND;


---
## 10 · The Production Kiosk — Cortex Agent + CoWork

A Streamlit prototype may work for a proof of concept (typically built by a Data Analyst to validate the idea with stakeholders), but for production you want a managed, governed interface — **Snowflake CoWork**.

**Why CoWork instead of a custom app?**
| Custom Streamlit App | Snowflake CoWork |
|---------------------|------------------|
| You maintain the code | Snowflake maintains the interface |
| You handle image rendering, formatting, errors | Built-in conversational UX |
| You manage deployment (stages, warehouses) | Already deployed — it's part of Snowsight |
| Access via Streamlit object grants | Access via Agent + role grants |

**What we'll do here:**
1. **Test** the agent you already provisioned (Section: Provision) to confirm it answers correctly
2. Access it through **CoWork** — no custom app needed
3. **Grant** access to the roles that should be able to use it

> :bulb: You already created the knowledge base, search service, and agent when you provisioned the stack at the top of this lab. This section does **not** rebuild them — it puts the governed agent in front of users.

### Step 1: Test the agent programmatically

Before opening it up to users, verify the agent works correctly using `DATA_AGENT_RUN` — the SQL function that calls an agent directly. This is how you **smoke-test** an agent from a notebook or script before anyone else sees it. It's not how end-users interact with it (that's CoWork, in Step 2).

In [ ]:
-- Test the agent with a coffee question
WITH raw_response AS (
  SELECT TRY_PARSE_JSON(
    SNOWFLAKE.CORTEX.DATA_AGENT_RUN(
      '{{my_db}}.RAW.{{user}}_COFFEE_MENU_AGENT',
      $${
        "messages": [
          {
            "role": "user",
            "content": [
              { "type": "text", "text": "Which coffee drinks have the most caffeine and why?" }
            ]
          }
        ]
      }$$,
      TRUE
    )
  ) AS response
)
SELECT f.value:"text"::STRING AS answer
FROM raw_response, LATERAL FLATTEN(input => response:"content") f
WHERE f.value:"type"::STRING = 'text';


### :jigsaw: Your turn — ask the agent
Change the question below to anything about the coffee menu and run it. This is the same [`DATA_AGENT_RUN`](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cortex-agents-manage) call your applications and CoWork make under the hood — notice the answer stays grounded in your knowledge base (that's the response guardrail you set at provisioning working).

In [ ]:
-- ✏️ Edit the question, then run
WITH raw_response AS (
  SELECT TRY_PARSE_JSON(
    SNOWFLAKE.CORTEX.DATA_AGENT_RUN(
      '{{my_db}}.RAW.{{user}}_COFFEE_MENU_AGENT',
      $${
        "messages": [
          { "role": "user", "content": [ { "type": "text", "text": "What is a good low-caffeine drink?" } ] }
        ]
      }$$,
      TRUE
    )
  ) AS response
)
SELECT f.value:"text"::STRING AS answer
FROM raw_response, LATERAL FLATTEN(input => response:"content") f
WHERE f.value:"type"::STRING = 'text';

### Step 2: Access via Snowflake CoWork

Your agent is now live. **Snowflake CoWork** provides a conversational interface that connects to your Cortex Agent — no custom app needed. This is how **end-users** will interact with the Coffee Kiosk agent in production.

> ⚠️ **CoWork ≠ CoCo.** The sparkle icon in your notebook's right panel is **CoCo** (Cortex Code — the coding assistant). **CoWork** is a separate product in the **Snowsight left sidebar** (outside the Workspace). You need to navigate out of the Workspace to reach it.

**To access CoWork:**
1. In the Snowsight left sidebar, hover/click **AI & ML** (the sparkle icon) to expand it, then **right-click Snowflake CoWork → open in a new tab** (so you keep this notebook open)
2. **Set your warehouse:** Click the **Settings** icon (gear) in CoWork → set **Warehouse** to `{{my_wh}}` → confirm. CoWork runs in its own session and won't inherit your notebook's warehouse — without this, agent calls will fail with a "no active warehouse" error.
3. Select **"Coffee Menu Expert ({{user}})"** — your username in the display name identifies which agent is yours
4. Ask a coffee question — CoWork routes it to your agent

**Try it:** Ask CoWork: *"What's the difference between a flat white and a latte?"*

> :bulb: **This is the production Coffee Kiosk.** The counter device just needs a browser pointed at CoWork. No custom app to maintain, no Streamlit deployment to manage. When the knowledge base is updated (new PDFs added), the search service refreshes automatically and CoWork immediately has access to the latest information.

### Step 3: Grant access to other roles

**Why this step matters:** Right now, only YOUR role (the owner) can see and use the agent. If the analyst team tried to open CoWork, they wouldn't even see the Coffee Kiosk agent — it's invisible to roles that don't have grants. As admin, you control who gets access by granting the full chain of privileges.

The cell below grants the analyst role (`{{my_analyst_role}}`) the full access chain — database → schema → search service → agent — then confirms it. Remember the owner's-rights caution from Section 2: granting `USAGE` on the search service exposes everything it indexed.

> 📌 **Why a chain and not just one grant?** Snowflake RBAC is hierarchical. To reach `{{user}}_COFFEE_MENU_AGENT`, a role needs `USAGE` on the database, `USAGE` on the schema, `USAGE` on the search service (it's a tool resource), and `USAGE` on the agent itself. Miss any link in the chain and the agent stays invisible.

In [ ]:
-- As the owner ({{my_role}}), grant the analyst role real access to the agent.
-- Cortex Search + Agent both need USAGE down the chain: database, schema, service, agent.
GRANT USAGE ON DATABASE {{my_db}} TO ROLE {{my_analyst_role}};
GRANT USAGE ON SCHEMA {{my_db}}.RAW TO ROLE {{my_analyst_role}};
GRANT USAGE ON CORTEX SEARCH SERVICE {{my_db}}.RAW.COFFEE_SEARCH_SVC TO ROLE {{my_analyst_role}};
GRANT USAGE ON AGENT {{my_db}}.RAW.{{user}}_COFFEE_MENU_AGENT TO ROLE {{my_analyst_role}};

-- Confirm the grant landed
SHOW GRANTS ON AGENT {{my_db}}.RAW.{{user}}_COFFEE_MENU_AGENT;


> :bulb: **What you just accomplished as an admin:**
>
> | What | Why it matters |
> |------|---------------|
> | Tested the governed agent | Confirmed it answers correctly before exposing it to users |
> | Connected to CoWork | Zero-maintenance interface — Snowflake handles the UX |
> | Controlled access via grants | Only authorized roles can use the kiosk |
>
> The Data Analyst proved the concept with a Streamlit prototype. You provisioned and governed the production version — safe, scalable, and accessible via CoWork without a single line of frontend code.

---
## Verify
If your lab ran successfully, you should have:

| Section | What you accomplished | How to confirm |
|---------|----------------------|----------------|
| Provision | Knowledge base, Search Service, Agent created | Section 1 verify cells show objects exist |
| Govern retrieval | Attribute-filtered search returns only in-scope rows | `preview_filtered_retrieval` returns Cold Drink only |
| Agent guardrails | Agent answers grounded and on-brand | `query_agent` answer cites the knowledge base |
| Access control | Analyst role granted USAGE on the agent | `SHOW GRANTS ON AGENT` lists `{{my_analyst_role}}` |
| Audit & eval | Agent answers logged and scored | `AGENT_ANSWER_LOG` has rows; grounded=~4/TRUE, absurd=~1/FALSE |
| PII Redaction | Customer names/emails replaced with placeholders | `AI_REDACT` output shows `[PERSON]`, `[EMAIL]` |
| Brand Safety | Inappropriate responses flagged as FALSE | `AI_FILTER` correctly blocked 2 of 4 test responses |
| Model Comparison | 8b vs 70b quality and token differences understood | Results show same tokens but different response quality |
| AI Consumption | Credit usage visible by service type and user | `METERING_DAILY_HISTORY` query returns data |
| Governed Wrapper | Full redact → answer → filter → log flow working | `GOVERNED_INTERACTIONS` table has at least 1 row |
| Cost Monitor | Task-based alert logging threshold breaches | `AI_COST_ALERTS` table has rows (threshold = 0.01) |
| Production Kiosk | Agent accessible via CoWork | CoWork can answer coffee questions |

**Objects in your database:**
- `{{my_db}}.RAW.COFFEE_KNOWLEDGE_BASE` — one row per coffee PDF (8 drinks; 9 if you also ran the Data Engineer lab), all classified
- `{{my_db}}.RAW.COFFEE_SEARCH_SVC` — Cortex Search Service (active)
- `{{my_db}}.RAW.{{user}}_COFFEE_MENU_AGENT` — Cortex Agent (responding via CoWork)
- `{{my_db}}.RAW.AGENT_ANSWER_LOG` — audit log of evaluated agent answers
- `{{my_db}}.RAW.GOVERNED_INTERACTIONS` — Audit log of governed interactions
- `{{my_db}}.RAW.AI_COST_ALERTS` — Threshold violation log
- `{{my_db}}.RAW.AI_COST_MONITOR_TASK` — Cost monitor (should be **suspended** by now)

---
### Check your understanding 🧠
A few questions on this lab's key themes. Run each quiz cell below, choose an answer, and review the feedback.

In [ ]:
from resources.knowledgecheck import platform_admin_quiz as quiz
quiz.display_question("q1")

In [ ]:
from resources.knowledgecheck import platform_admin_quiz as quiz
quiz.display_question("q2")

In [ ]:
from resources.knowledgecheck import platform_admin_quiz as quiz
quiz.display_question("q3")

In [ ]:
from resources.knowledgecheck import platform_admin_quiz as quiz
quiz.display_question("q4")

---
### ❄️ Key Takeaways
- **Provision then verify:** stand up the KB → search → agent stack, then confirm it before go-live
- **Guardrails live in the agent spec:** response instructions (grounding, on-brand), orchestration, and a budget
- **Retrieval governance:** attribute filters scope what's retrievable; Cortex Search runs with owner's rights — grant it deliberately
- **Reuse Lab 5's controls:** AI_REDACT / AI_FILTER now guard the agent's inputs and outputs
- **Access control:** grant USAGE down the chain (db → schema → service → agent) to the roles that need it
- **Audit & evaluate:** log agent answers and score them (LLM-as-judge) so drift is caught automatically
- The admin's job is to make an AI agent safe, grounded, and affordable BEFORE it reaches customers

**The Complete Picture — Three Personas:**
- The **Data Analyst** proved the concept (Streamlit prototype, CoCo-assisted)
- The **Data Engineer** built the scalable pipeline (parse, embed, search, agent)
- **You (Admin)** governed it and made it production-ready via CoWork
- Together: a complete, governed, customer-facing AI system

---
### Optional Extra: Access via Snowflake Mobile App

You can also access your Coffee Kiosk agent from your phone using the **Snowflake Mobile App**. This gives you the same CoWork conversational interface on the go — useful for quick demos or testing outside the office.

**Setup steps:**
1. Download the **Snowflake CoWork** app from the [App Store (iOS)](https://apps.apple.com/us/app/snowflake-cowork/id6755540372). *(Snowflake CoWork is currently **iOS-only** — it is not on Google Play.)*
2. Open the app and enter your **account identifier** — find it in Snowsight under your account menu (bottom-left), or as the `<orgname>-<account>` value in your Snowsight URL
3. Log in with the same credentials you use for Snowsight
4. Start a conversation and ask a coffee question — your agent responds just like in the browser

> :warning: **MFA note:** CoWork honors your account's MFA. On an MFA-enabled account you must have an MFA method that works **on the phone itself** (e.g. a passkey registered on the phone, or an authenticator app). If your only MFA method lives on another device (e.g. a passkey in your laptop browser), phone login will fail until you register a phone-accessible method.

> :bulb: **Why this matters for admins:** The mobile app uses the same role-based access controls as Snowsight. If a user's role doesn't have access to the agent, they won't see it on mobile either. No additional configuration needed — your governance controls carry over automatically.

> :warning: **This is optional.** Everything you need for this lab works in the browser via CoWork. The mobile app is simply another access channel for the same governed agent.

### Notebook completion. 🏁
You're done — your notebook is saved automatically.

> 💡 **Moving on to another lab?** Shut down this notebook's **kernel** — click **Connected** at the top → **Shut down kernel**. All your lab notebooks share one compute service, so clearing a finished kernel frees up memory for the next lab. Your service stays running, ready in seconds.

> 💡 **Finished for the day?** **Suspend your service** — click **Connected** → hover your service name → **Suspend**. This stops compute; it resumes automatically next time you run a cell.